<a href="https://colab.research.google.com/github/minju0236/Hankyung-Bootcamp/blob/main/Day6_5_(260615)_HTTPS%2C_CORS_%EC%A0%95%EC%B1%85_%EB%B0%8F_%EC%84%B8%EC%85%98_vs_JWT_%EC%9D%B8%EC%A6%9D_%EB%B0%A9%EC%8B%9D_%EB%B9%84%EA%B5%90_%EB%B6%84%EC%84%9D_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile /content/security-jwt/security-jpa-jwt/build.gradle

plugins {
    id 'java'
    id 'org.springframework.boot' version '3.3.5'
    id 'io.spring.dependency-management' version '1.1.6'
}

group = 'com.example'
version = '0.0.1-SNAPSHOT'

java {
    toolchain {
        languageVersion = JavaLanguageVersion.of(21)
    }
}

repositories {
    mavenCentral()
}

dependencies {
    implementation 'org.springframework.boot:spring-boot-starter-web'
    implementation 'org.springframework.boot:spring-boot-starter-security'
    implementation 'org.springframework.boot:spring-boot-starter-data-jpa'

    runtimeOnly 'org.mariadb.jdbc:mariadb-java-client'

    implementation 'io.jsonwebtoken:jjwt-api:0.12.6'
    runtimeOnly 'io.jsonwebtoken:jjwt-impl:0.12.6'
    runtimeOnly 'io.jsonwebtoken:jjwt-jackson:0.12.6'

    testImplementation 'org.springframework.boot:spring-boot-starter-test'
    testImplementation 'org.springframework.security:spring-security-test'
    testRuntimeOnly 'org.junit.platform:junit-platform-launcher'
}

tasks.named('test') {
    useJUnitPlatform()
}

Overwriting /content/security-jwt/security-jpa-jwt/build.gradle


In [ ]:
%%writefile /content/security-jwt/security-jpa-jwt/src/main/resources/application.properties

server.port=3100
spring.application.name=security-jpa-jwt

spring.datasource.url=jdbc:mariadb://localhost:3306/test_db
spring.datasource.username=testuser
spring.datasource.password=1234
spring.datasource.driver-class-name=org.mariadb.jdbc.Driver

spring.jpa.hibernate.ddl-auto=update
spring.jpa.show-sql=true
spring.jpa.properties.hibernate.format_sql=true

jwt.secret=0123456789012345678901234567890123456789012345678901234567890123
jwt.expiration-ms=3600000

logging.level.org.springframework.security=DEBUG

Overwriting /content/security-jwt/security-jpa-jwt/src/main/resources/application.properties


In [ ]:
%%writefile /content/security-jwt/security-jpa-jwt/src/main/java/com/example/demo/SecurityJpaJwtApplication.java

package com.example.demo;

import org.springframework.boot.SpringApplication;
import org.springframework.boot.autoconfigure.SpringBootApplication;

@SpringBootApplication
public class SecurityJpaJwtApplication {

    public static void main(String[] args) {
        SpringApplication.run(SecurityJpaJwtApplication.class, args);
    }
}

Overwriting /content/security-jwt/security-jpa-jwt/src/main/java/com/example/demo/SecurityJpaJwtApplication.java


In [ ]:
%%writefile /content/security-jwt/security-jpa-jwt/src/main/java/com/example/demo/entity/UserRole.java

package com.example.demo.entity;

public enum UserRole {
    USER,
    ADMIN
}

Writing /content/security-jwt/security-jpa-jwt/src/main/java/com/example/demo/entity/UserRole.java


In [ ]:
%%writefile /content/security-jwt/security-jpa-jwt/src/main/java/com/example/demo/entity/AppUser.java

package com.example.demo.entity;

import jakarta.persistence.*;

import java.time.LocalDateTime;

@Entity
@Table(
        name = "users",
        uniqueConstraints = {
                @UniqueConstraint(name = "uk_users_email", columnNames = "email")
        }
)
public class AppUser {

    @Id
    @GeneratedValue(strategy = GenerationType.IDENTITY)
    private Long id;

    @Column(nullable = false, length = 100)
    private String email;

    @Column(nullable = false, length = 255)
    private String password;

    @Enumerated(EnumType.STRING)
    @Column(nullable = false, length = 20)
    private UserRole role;

    @Column(nullable = false)
    private LocalDateTime createdAt;

    protected AppUser() {
    }

    public AppUser(String email, String password, UserRole role) {
        this.email = email;
        this.password = password;
        this.role = role;
        this.createdAt = LocalDateTime.now();
    }

    public Long getId() {
        return id;
    }

    public String getEmail() {
        return email;
    }

    public String getPassword() {
        return password;
    }

    public UserRole getRole() {
        return role;
    }

    public LocalDateTime getCreatedAt() {
        return createdAt;
    }
}

Writing /content/security-jwt/security-jpa-jwt/src/main/java/com/example/demo/entity/AppUser.java


In [ ]:
%%writefile /content/security-jwt/security-jpa-jwt/src/main/java/com/example/demo/repository/AppUserRepository.java

package com.example.demo.repository;

import com.example.demo.entity.AppUser;
import org.springframework.data.jpa.repository.JpaRepository;

import java.util.Optional;

public interface AppUserRepository extends JpaRepository<AppUser, Long> {

    Optional<AppUser> findByEmail(String email);

    boolean existsByEmail(String email);
}

Writing /content/security-jwt/security-jpa-jwt/src/main/java/com/example/demo/repository/AppUserRepository.java


In [ ]:
%%writefile /content/security-jwt/security-jpa-jwt/src/main/java/com/example/demo/dto/SignupRequest.java

package com.example.demo.dto;

public class SignupRequest {

    private String email;
    private String password;
    private String role;

    public SignupRequest() {
    }

    public String getEmail() {
        return email;
    }

    public String getPassword() {
        return password;
    }

    public String getRole() {
        return role;
    }
}

Writing /content/security-jwt/security-jpa-jwt/src/main/java/com/example/demo/dto/SignupRequest.java


In [ ]:
%%writefile /content/security-jwt/security-jpa-jwt/src/main/java/com/example/demo/dto/LoginRequest.java

package com.example.demo.dto;

public class LoginRequest {

    private String email;
    private String password;

    public LoginRequest() {
    }

    public String getEmail() {
        return email;
    }

    public String getPassword() {
        return password;
    }
}

Writing /content/security-jwt/security-jpa-jwt/src/main/java/com/example/demo/dto/LoginRequest.java


In [ ]:
%%writefile /content/security-jwt/security-jpa-jwt/src/main/java/com/example/demo/dto/LoginResponse.java

package com.example.demo.dto;

public class LoginResponse {

    private String tokenType;
    private String accessToken;
    private String email;
    private String role;
    private long expiresInMs;

    public LoginResponse(String accessToken, String email, String role, long expiresInMs) {
        this.tokenType = "Bearer";
        this.accessToken = accessToken;
        this.email = email;
        this.role = role;
        this.expiresInMs = expiresInMs;
    }

    public String getTokenType() {
        return tokenType;
    }

    public String getAccessToken() {
        return accessToken;
    }

    public String getEmail() {
        return email;
    }

    public String getRole() {
        return role;
    }

    public long getExpiresInMs() {
        return expiresInMs;
    }
}

Writing /content/security-jwt/security-jpa-jwt/src/main/java/com/example/demo/dto/LoginResponse.java


In [ ]:
%%writefile /content/security-jwt/security-jpa-jwt/src/main/java/com/example/demo/dto/MeResponse.java

package com.example.demo.dto;

import java.util.Collection;

public class MeResponse {

    private String email;
    private Collection<?> authorities;
    private String message;

    public MeResponse(String email, Collection<?> authorities, String message) {
        this.email = email;
        this.authorities = authorities;
        this.message = message;
    }

    public String getEmail() {
        return email;
    }

    public Collection<?> getAuthorities() {
        return authorities;
    }

    public String getMessage() {
        return message;
    }
}

Writing /content/security-jwt/security-jpa-jwt/src/main/java/com/example/demo/dto/MeResponse.java


In [ ]:
%%writefile /content/security-jwt/security-jpa-jwt/src/main/java/com/example/demo/dto/AuthCompareResponse.java

package com.example.demo.dto;

public class AuthCompareResponse {

    private String mode;
    private String storage;
    private String clientSend;
    private String serverCheck;
    private String logout;
    private String suitableFor;

    public AuthCompareResponse(
            String mode,
            String storage,
            String clientSend,
            String serverCheck,
            String logout,
            String suitableFor
    ) {
        this.mode = mode;
        this.storage = storage;
        this.clientSend = clientSend;
        this.serverCheck = serverCheck;
        this.logout = logout;
        this.suitableFor = suitableFor;
    }

    public String getMode() {
        return mode;
    }

    public String getStorage() {
        return storage;
    }

    public String getClientSend() {
        return clientSend;
    }

    public String getServerCheck() {
        return serverCheck;
    }

    public String getLogout() {
        return logout;
    }

    public String getSuitableFor() {
        return suitableFor;
    }
}

Writing /content/security-jwt/security-jpa-jwt/src/main/java/com/example/demo/dto/AuthCompareResponse.java


In [ ]:
%%writefile /content/security-jwt/security-jpa-jwt/src/main/java/com/example/demo/security/CustomUserDetailsService.java

package com.example.demo.security;

import com.example.demo.entity.AppUser;
import com.example.demo.repository.AppUserRepository;
import org.springframework.security.core.userdetails.User;
import org.springframework.security.core.userdetails.UserDetails;
import org.springframework.security.core.userdetails.UserDetailsService;
import org.springframework.security.core.userdetails.UsernameNotFoundException;
import org.springframework.stereotype.Service;
import org.springframework.transaction.annotation.Transactional;

@Service
public class CustomUserDetailsService implements UserDetailsService {

    private final AppUserRepository appUserRepository;

    public CustomUserDetailsService(AppUserRepository appUserRepository) {
        this.appUserRepository = appUserRepository;
    }

    @Override
    @Transactional(readOnly = true)
    public UserDetails loadUserByUsername(String email) throws UsernameNotFoundException {
        AppUser appUser = appUserRepository.findByEmail(email)
                .orElseThrow(() -> new UsernameNotFoundException("사용자를 찾을 수 없습니다: " + email));

        return User.builder()
                .username(appUser.getEmail())
                .password(appUser.getPassword())
                .roles(appUser.getRole().name())
                .build();
    }
}

Writing /content/security-jwt/security-jpa-jwt/src/main/java/com/example/demo/security/CustomUserDetailsService.java


In [ ]:
%%writefile /content/security-jwt/security-jpa-jwt/src/main/java/com/example/demo/security/JwtTokenProvider.java

package com.example.demo.security;

import io.jsonwebtoken.Claims;
import io.jsonwebtoken.Jwts;
import io.jsonwebtoken.security.Keys;
import org.springframework.beans.factory.annotation.Value;
import org.springframework.security.core.Authentication;
import org.springframework.security.core.GrantedAuthority;
import org.springframework.stereotype.Component;

import javax.crypto.SecretKey;
import java.nio.charset.StandardCharsets;
import java.util.Date;
import java.util.stream.Collectors;

@Component
public class JwtTokenProvider {

    private final SecretKey secretKey;
    private final long expirationMs;

    public JwtTokenProvider(
            @Value("${jwt.secret}") String secret,
            @Value("${jwt.expiration-ms}") long expirationMs
    ) {
        this.secretKey = Keys.hmacShaKeyFor(secret.getBytes(StandardCharsets.UTF_8));
        this.expirationMs = expirationMs;
    }

    public String createToken(Authentication authentication) {
        String email = authentication.getName();

        String role = authentication.getAuthorities()
                .stream()
                .map(GrantedAuthority::getAuthority)
                .collect(Collectors.joining(","));

        Date now = new Date();
        Date expiryDate = new Date(now.getTime() + expirationMs);

        return Jwts.builder()
                .subject(email)
                .claim("role", role)
                .issuedAt(now)
                .expiration(expiryDate)
                .signWith(secretKey)
                .compact();
    }

    public String getEmail(String token) {
        return parseClaims(token).getSubject();
    }

    public String getRole(String token) {
        return parseClaims(token).get("role", String.class);
    }

    public boolean validateToken(String token) {
        parseClaims(token);
        return true;
    }

    public long getExpirationMs() {
        return expirationMs;
    }

    private Claims parseClaims(String token) {
        return Jwts.parser()
                .verifyWith(secretKey)
                .build()
                .parseSignedClaims(token)
                .getPayload();
    }
}

Writing /content/security-jwt/security-jpa-jwt/src/main/java/com/example/demo/security/JwtTokenProvider.java


In [ ]:
%%writefile /content/security-jwt/security-jpa-jwt/src/main/java/com/example/demo/security/JwtAuthenticationFilter.java

package com.example.demo.security;

import jakarta.servlet.FilterChain;
import jakarta.servlet.ServletException;
import jakarta.servlet.http.HttpServletRequest;
import jakarta.servlet.http.HttpServletResponse;
import org.springframework.security.authentication.UsernamePasswordAuthenticationToken;
import org.springframework.security.core.context.SecurityContextHolder;
import org.springframework.security.core.userdetails.UserDetails;
import org.springframework.security.web.authentication.WebAuthenticationDetailsSource;
import org.springframework.stereotype.Component;
import org.springframework.util.StringUtils;
import org.springframework.web.filter.OncePerRequestFilter;

import java.io.IOException;

@Component
public class JwtAuthenticationFilter extends OncePerRequestFilter {

    private final JwtTokenProvider jwtTokenProvider;
    private final CustomUserDetailsService customUserDetailsService;

    public JwtAuthenticationFilter(
            JwtTokenProvider jwtTokenProvider,
            CustomUserDetailsService customUserDetailsService
    ) {
        this.jwtTokenProvider = jwtTokenProvider;
        this.customUserDetailsService = customUserDetailsService;
    }

    @Override
    protected void doFilterInternal(
            HttpServletRequest request,
            HttpServletResponse response,
            FilterChain filterChain
    ) throws ServletException, IOException {
        String token = resolveToken(request);

        if (token != null && jwtTokenProvider.validateToken(token)) {
            String email = jwtTokenProvider.getEmail(token);

            UserDetails userDetails = customUserDetailsService.loadUserByUsername(email);

            UsernamePasswordAuthenticationToken authentication =
                    new UsernamePasswordAuthenticationToken(
                            userDetails,
                            null,
                            userDetails.getAuthorities()
                    );

            authentication.setDetails(
                    new WebAuthenticationDetailsSource().buildDetails(request)
            );

            SecurityContextHolder.getContext().setAuthentication(authentication);
        }

        filterChain.doFilter(request, response);
    }

    private String resolveToken(HttpServletRequest request) {
        String bearerToken = request.getHeader("Authorization");

        if (StringUtils.hasText(bearerToken) && bearerToken.startsWith("Bearer ")) {
            return bearerToken.substring(7);
        }

        return null;
    }
}

Writing /content/security-jwt/security-jpa-jwt/src/main/java/com/example/demo/security/JwtAuthenticationFilter.java


In [ ]:
%%writefile /content/security-jwt/security-jpa-jwt/src/main/java/com/example/demo/config/SecurityConfig.java

package com.example.demo.config;

import com.example.demo.security.JwtAuthenticationFilter;
import org.springframework.context.annotation.Bean;
import org.springframework.context.annotation.Configuration;
import org.springframework.security.authentication.AuthenticationManager;
import org.springframework.security.config.annotation.authentication.configuration.AuthenticationConfiguration;
import org.springframework.security.config.annotation.method.configuration.EnableMethodSecurity;
import org.springframework.security.config.annotation.web.builders.HttpSecurity;
import org.springframework.security.config.annotation.web.configuration.EnableWebSecurity;
import org.springframework.security.config.http.SessionCreationPolicy;
import org.springframework.security.crypto.bcrypt.BCryptPasswordEncoder;
import org.springframework.security.crypto.password.PasswordEncoder;
import org.springframework.security.web.SecurityFilterChain;
import org.springframework.security.web.authentication.UsernamePasswordAuthenticationFilter;

@Configuration
@EnableWebSecurity
@EnableMethodSecurity
public class SecurityConfig {

    private final JwtAuthenticationFilter jwtAuthenticationFilter;

    public SecurityConfig(JwtAuthenticationFilter jwtAuthenticationFilter) {
        this.jwtAuthenticationFilter = jwtAuthenticationFilter;
    }

    @Bean
    public SecurityFilterChain securityFilterChain(HttpSecurity http) throws Exception {
        http
                .cors(cors -> {})
                .csrf(csrf -> csrf.disable())
                .sessionManagement(session -> session
                        .sessionCreationPolicy(SessionCreationPolicy.STATELESS)
                )
                .authorizeHttpRequests(auth -> auth
                        .requestMatchers(
                                "/api/public",
                                "/api/auth/signup",
                                "/api/auth/login",
                                "/api/auth/session-vs-jwt"
                        ).permitAll()
                        .requestMatchers("/api/admin").hasRole("ADMIN")
                        .anyRequest().authenticated()
                )
                .addFilterBefore(
                        jwtAuthenticationFilter,
                        UsernamePasswordAuthenticationFilter.class
                );

        return http.build();
    }

    @Bean
    public AuthenticationManager authenticationManager(
            AuthenticationConfiguration authenticationConfiguration
    ) throws Exception {
        return authenticationConfiguration.getAuthenticationManager();
    }

    @Bean
    public PasswordEncoder passwordEncoder() {
        return new BCryptPasswordEncoder();
    }
}

Writing /content/security-jwt/security-jpa-jwt/src/main/java/com/example/demo/config/SecurityConfig.java


In [ ]:
%%writefile /content/security-jwt/security-jpa-jwt/src/main/java/com/example/demo/config/CorsConfig.java

package com.example.demo.config;

import org.springframework.context.annotation.Bean;
import org.springframework.context.annotation.Configuration;
import org.springframework.web.cors.CorsConfiguration;
import org.springframework.web.cors.CorsConfigurationSource;
import org.springframework.web.cors.UrlBasedCorsConfigurationSource;

import java.util.List;

@Configuration
public class CorsConfig {

    @Bean
    public CorsConfigurationSource corsConfigurationSource() {
        CorsConfiguration configuration = new CorsConfiguration();

        configuration.setAllowedOrigins(List.of("http://localhost:3000"));
        configuration.setAllowedMethods(List.of("GET", "POST", "PUT", "PATCH", "DELETE", "OPTIONS"));
        configuration.setAllowedHeaders(List.of("Authorization", "Content-Type"));
        configuration.setAllowCredentials(true);

        UrlBasedCorsConfigurationSource source = new UrlBasedCorsConfigurationSource();
        source.registerCorsConfiguration("/api/**", configuration);

        return source;
    }
}


Writing /content/security-jwt/security-jpa-jwt/src/main/java/com/example/demo/config/CorsConfig.java


In [ ]:
%%writefile /content/security-jwt/security-jpa-jwt/src/main/java/com/example/demo/controller/AuthController.java

package com.example.demo.controller;

import com.example.demo.dto.AuthCompareResponse;
import com.example.demo.dto.LoginRequest;
import com.example.demo.dto.LoginResponse;
import com.example.demo.dto.SignupRequest;
import com.example.demo.entity.AppUser;
import com.example.demo.entity.UserRole;
import com.example.demo.repository.AppUserRepository;
import com.example.demo.security.JwtTokenProvider;
import org.springframework.http.HttpStatus;
import org.springframework.http.ResponseEntity;
import org.springframework.security.authentication.AuthenticationManager;
import org.springframework.security.authentication.UsernamePasswordAuthenticationToken;
import org.springframework.security.core.Authentication;
import org.springframework.security.crypto.password.PasswordEncoder;
import org.springframework.web.bind.annotation.*;

@RestController
@RequestMapping("/api/auth")
public class AuthController {

    private final AppUserRepository appUserRepository;
    private final PasswordEncoder passwordEncoder;
    private final AuthenticationManager authenticationManager;
    private final JwtTokenProvider jwtTokenProvider;

    public AuthController(
            AppUserRepository appUserRepository,
            PasswordEncoder passwordEncoder,
            AuthenticationManager authenticationManager,
            JwtTokenProvider jwtTokenProvider
    ) {
        this.appUserRepository = appUserRepository;
        this.passwordEncoder = passwordEncoder;
        this.authenticationManager = authenticationManager;
        this.jwtTokenProvider = jwtTokenProvider;
    }

    @PostMapping("/signup")
    public ResponseEntity<?> signup(@RequestBody SignupRequest request) {
        if (appUserRepository.existsByEmail(request.getEmail())) {
            return ResponseEntity.status(HttpStatus.CONFLICT)
                    .body("이미 사용 중인 이메일입니다.");
        }

        UserRole role = parseRole(request.getRole());

        String encodedPassword = passwordEncoder.encode(request.getPassword());

        AppUser appUser = new AppUser(
                request.getEmail(),
                encodedPassword,
                role
        );

        appUserRepository.save(appUser);

        return ResponseEntity.status(HttpStatus.CREATED)
                .body("회원가입이 완료되었습니다.");
    }

    @PostMapping("/login")
    public LoginResponse login(@RequestBody LoginRequest request) {
        Authentication authentication = authenticationManager.authenticate(
                new UsernamePasswordAuthenticationToken(
                        request.getEmail(),
                        request.getPassword()
                )
        );

        String accessToken = jwtTokenProvider.createToken(authentication);

        String role = authentication.getAuthorities()
                .iterator()
                .next()
                .getAuthority();

        return new LoginResponse(
                accessToken,
                authentication.getName(),
                role,
                jwtTokenProvider.getExpirationMs()
        );
    }

    @GetMapping("/session-vs-jwt")
    public AuthCompareResponse sessionVsJwt() {
        return new AuthCompareResponse(
                "Session vs JWT",
                "Session은 서버 저장소, JWT는 클라이언트 토큰 중심",
                "Session은 JSESSIONID 쿠키, JWT는 Authorization Bearer Header",
                "Session은 서버 세션 조회, JWT는 서명과 만료시간 검증",
                "Session은 서버 세션 삭제, JWT는 토큰 삭제와 만료·폐기 전략 필요",
                "Session은 내부 업무 시스템, JWT는 SPA·모바일·외부 API에 적합"
        );
    }

    private UserRole parseRole(String role) {
        if (role == null || role.isBlank()) {
            return UserRole.USER;
        }

        try {
            return UserRole.valueOf(role.toUpperCase());
        } catch (IllegalArgumentException e) {
            return UserRole.USER;
        }
    }
}

Writing /content/security-jwt/security-jpa-jwt/src/main/java/com/example/demo/controller/AuthController.java


In [ ]:
%%writefile /content/security-jwt/security-jpa-jwt/src/main/java/com/example/demo/controller/ApiController.java

package com.example.demo.controller;

import com.example.demo.dto.MeResponse;
import org.springframework.security.core.Authentication;
import org.springframework.web.bind.annotation.GetMapping;
import org.springframework.web.bind.annotation.RestController;

@RestController
public class ApiController {

    @GetMapping("/api/public")
    public String publicApi() {
        return "인증 없이 접근 가능한 공개 API입니다.";
    }

    @GetMapping("/api/me")
    public MeResponse me(Authentication authentication) {
        return new MeResponse(
                authentication.getName(),
                authentication.getAuthorities(),
                "인증된 사용자입니다."
        );
    }

    @GetMapping("/api/admin")
    public String adminApi(Authentication authentication) {
        return "관리자 API 접근 성공: " + authentication.getName();
    }
}

Writing /content/security-jwt/security-jpa-jwt/src/main/java/com/example/demo/controller/ApiController.java


In [ ]:
%%writefile /content/security-jwt/security-jpa-jwt/src/main/java/com/example/demo/config/DataInitializer.java

package com.example.demo.config;

import com.example.demo.entity.AppUser;
import com.example.demo.entity.UserRole;
import com.example.demo.repository.AppUserRepository;
import org.springframework.boot.CommandLineRunner;
import org.springframework.context.annotation.Bean;
import org.springframework.context.annotation.Configuration;
import org.springframework.security.crypto.password.PasswordEncoder;

@Configuration
public class DataInitializer {

    @Bean
    public CommandLineRunner initAdminUser(
            AppUserRepository appUserRepository,
            PasswordEncoder passwordEncoder
    ) {
        return args -> {
            String adminEmail = "admin@test.com";

            if (!appUserRepository.existsByEmail(adminEmail)) {
                AppUser admin = new AppUser(
                        adminEmail,
                        passwordEncoder.encode("1234"),
                        UserRole.ADMIN
                );

                appUserRepository.save(admin);
            }
        };
    }
}

Writing /content/security-jwt/security-jpa-jwt/src/main/java/com/example/demo/config/DataInitializer.java
